<a href="https://colab.research.google.com/github/Lakshay-Juneja/dva_project/blob/data/ipl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

new start from here

In [ ]:
df = pd.read_excel("/IPL.xlsx")
df.head()

,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [ ]:
df.columns = df.columns.str.lower().str.strip()

df['runs_total'] = pd.to_numeric(df['runs_total'], errors='coerce')
df['runs_batter'] = pd.to_numeric(df['runs_batter'], errors='coerce')

df['is_wicket'] = df['wicket_kind'].notna().astype(int)

df.head()

,unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out,is_wicket
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0


In [ ]:
df.columns = df.columns.str.lower().str.strip()

# Convert numeric columns
num_cols = ['runs_batter','runs_extras','runs_total','season','year']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Wicket flag
df['is_wicket'] = df['wicket_kind'].notna().astype(int)

overview_matches = pd.DataFrame({
    "Metric": ["Total Matches"],
    "Value": [df['match_id'].nunique()]
})

overview_matches


,Metric,Value
0,Total Matches,1169


In [ ]:
overview_matches = pd.DataFrame({
    "Metric": ["Total Matches"],
    "Value": [df['match_id'].nunique()]
})
overview_matches

,Metric,Value
0,Total Matches,1169


In [ ]:
overview_runs = pd.DataFrame({
    "Metric": ["Total Runs"],
    "Value": [df['runs_total'].sum()]
})
overview_runs

,Metric,Value
0,Total Runs,374283


In [ ]:
fours = (df['runs_batter'] == 4).sum()
sixes = (df['runs_batter'] == 6).sum()

overview_boundaries = pd.DataFrame({
    "Metric": ["Fours","Sixes"],
    "Value": [fours, sixes]
})
overview_boundaries

,Metric,Value
0,Fours,32113
1,Sixes,14353


In [ ]:
overview_teams = pd.DataFrame({
    "Metric":["Total Teams"],
    "Value":[df['batting_team'].nunique()]
})
overview_teams

,Metric,Value
0,Total Teams,19


In [ ]:
def era_map(season):
    if 2008 <= season <= 2012:
        return "2008-2012"
    elif 2013 <= season <= 2017:
        return "2013-2017"
    elif 2018 <= season <= 2022:
        return "2018-2022"
    else:
        return "2023-2025"

df['era'] = df['season'].apply(era_map)
print(era_map(2010))

2008-2012


In [ ]:
match_runs = df.groupby(['era','match_id'])['runs_total'].sum().reset_index()

era_avg_score = match_runs.groupby('era')['runs_total'].mean().reset_index()
era_avg_score.rename(columns={'runs_total':'avg_match_score'}, inplace=True)
match_runs
era_avg_score

,era,avg_match_score
0,2008-2012,293.921569
1,2013-2017,310.617834
2,2018-2022,324.279528
3,2023-2025,338.594458


In [ ]:
balls = df.groupby(['era','match_id']).size().reset_index(name='balls')
runs = df.groupby(['era','match_id'])['runs_total'].sum().reset_index()

rr = pd.merge(runs, balls, on=['era','match_id'])
rr['run_rate'] = (rr['runs_total']/rr['balls'])*6

era_runrate = rr.groupby('era')['run_rate'].mean().reset_index()
balls
runs
rr

,era,match_id,runs_total,balls,run_rate
0,2008-2012,392181,311,244,7.647541
1,2008-2012,392182,191,221,5.185520
2,2008-2012,392183,162,108,9.000000
3,2008-2012,392184,205,199,6.180905
4,2008-2012,392185,266,220,7.254545
...,...,...,...,...,...
1164,2023-2025,1473508,207,147,8.448980
1165,2023-2025,1473509,436,248,10.548387
1166,2023-2025,1473510,410,244,10.081967
1167,2023-2025,1473511,374,252,8.904762


In [ ]:
df['is_six'] = (df['runs_batter'] == 6).astype(int)

era_sixes = df.groupby('era')['is_six'].sum().reset_index()
era_sixes

,era,is_six
0,2008-2012,1880
1,2013-2017,3433
2,2018-2022,3407
3,2023-2025,5633


In [ ]:
era_wickets = df.groupby('era')['is_wicket'].sum().reset_index()
era_wickets

,era,is_wicket
0,2008-2012,2369
1,2013-2017,3654
2,2018-2022,3036
3,2023-2025,4764


In [ ]:
team_season_runs = df.groupby(
    ['season','batting_team']
)['runs_total'].sum().reset_index()
team_season_runs

,season,batting_team,runs_total
0,2009.0,Chennai Super Kings,2231
1,2009.0,Deccan Chargers,2408
2,2009.0,Delhi Daredevils,2131
3,2009.0,Kings XI Punjab,1928
4,2009.0,Kolkata Knight Riders,1772
...,...,...,...
127,2025.0,Mumbai Indians,2912
128,2025.0,Punjab Kings,3262
129,2025.0,Rajasthan Royals,2614
130,2025.0,Royal Challengers Bengaluru,2653


In [ ]:
team_wickets = df.groupby(
    ['season','bowling_team']
)['is_wicket'].sum().reset_index()
team_wickets

,season,bowling_team,is_wicket
0,2009.0,Chennai Super Kings,91
1,2009.0,Deccan Chargers,110
2,2009.0,Delhi Daredevils,106
3,2009.0,Kings XI Punjab,76
4,2009.0,Kolkata Knight Riders,59
...,...,...,...
127,2025.0,Mumbai Indians,116
128,2025.0,Punjab Kings,97
129,2025.0,Rajasthan Royals,68
130,2025.0,Royal Challengers Bengaluru,94


In [ ]:
team_match_runs = df.groupby(
    ['batting_team','match_id']
)['runs_total'].sum().reset_index()

team_avg_score = team_match_runs.groupby(
    'batting_team'
)['runs_total'].mean().reset_index()
team_match_runs
team_avg_score

,batting_team,runs_total
0,Chennai Super Kings,163.625498
1,Deccan Chargers,152.840000
2,Delhi Capitals,165.714286
3,Delhi Daredevils,150.906832
4,Gujarat Lions,162.066667
5,Gujarat Titans,177.483333
6,Kings XI Punjab,158.231579
7,Kochi Tuskers Kerala,135.785714
8,Kolkata Knight Riders,156.564394
9,Lucknow Super Giants,176.586207


In [ ]:
team_boundaries = df[df['runs_batter'].isin([4,6])] \
    .groupby('batting_team')['runs_batter'] \
    .count() \
    .reset_index(name='boundary_count')
team_boundaries

,batting_team,boundary_count
0,Chennai Super Kings,5006
1,Deccan Chargers,1357
2,Delhi Capitals,2233
3,Delhi Daredevils,2957
4,Gujarat Lions,615
5,Gujarat Titans,1342
6,Kings XI Punjab,3706
7,Kochi Tuskers Kerala,223
8,Kolkata Knight Riders,5230
9,Lucknow Super Giants,1279


In [ ]:
top_batters = df.groupby('batter')['runs_batter'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10) \
    .reset_index()
top_batters

,batter,runs_batter
0,V Kohli,8671
1,RG Sharma,7048
2,S Dhawan,6769
3,DA Warner,6567
4,SK Raina,5536
5,MS Dhoni,5439
6,KL Rahul,5235
7,AB de Villiers,5181
8,AM Rahane,5032
9,CH Gayle,4997


In [ ]:
top_bowlers = df.groupby('bowler')['is_wicket'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10) \
    .reset_index()
top_bowlers

,bowler,is_wicket
0,YS Chahal,229
1,B Kumar,213
2,SP Narine,212
3,DJ Bravo,207
4,R Ashwin,205
5,JJ Bumrah,203
6,PP Chawla,201
7,SL Malinga,188
8,A Mishra,183
9,RA Jadeja,179
